# REVENUE INDEX — MULTI-SCENARIO (3 penetration × 3 temperature = 9 combinations)
=================================================================================
Computes RI(t) = P(t) · CR_sim(t) for all 9 scenario combinations.
 
Penetration scenarios (from mc_physical_variables_{key}.parquet):
  current_trend  → 30% solar share by 2050
  pniec          → 45% solar share by 2050  (PNIEC 2024)
  entso_e        → 60% solar share by 2050  (ENTSO-E DE)
 
Temperature scenarios (SSP delta from df_temperature_scenarios.parquet):
  baseline  → historical trend only   (no SSP adjustment)
  ssp126    → aggressive mitigation   (delta < 0 vs baseline)
  ssp585    → high emissions          (delta > 0 vs baseline)
 
Reference scenario for value-loss analysis: (pniec, baseline)
 
Steps:
  Step 0  — Imports, paths, constants, combination matrix
  Step 1  — NOCT uplift profile
  Step 2  — Load shared arrays (GHI_sim, T_sim) + SSP deltas
  Step 3  — Compute T_eff, f_temp, P(t) for each temp scenario [×3]
  Step 4  — Compute RI(t) for all 9 combinations
  Step 5  — Aggregate to annual + risk metrics
  Step 6  — Save results
  Step 7  — Individual fan charts (3×3 grid)
  Step 8  — Cross-scenario comparison plots
  Step 9  — Waterfall decomposition of Revenue-at-Risk

In [ ]:
 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from scipy.stats import gaussian_kde
from pathlib import Path
import pickle
import calendar
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# ── paths ─────────────────────────────────────────────────────────────────────
data_sim      = Path("../../Data/Simulated")
data_modelled = Path("../../Data/Modelled")
data_cleaned  = Path("../../Data/Cleaned")
data_results  = Path("../../Data/Results")
models_path   = Path("../../Code/Models")
data_results.mkdir(parents=True, exist_ok=True)

In [ ]:
# ── physical constants ────────────────────────────────────────────────────────
G_STC  = 1000.0   # W/m²
T_STC  = 25.0     # °C
GAMMA  = -0.0043   # /°C
PR     = 0.80
NOCT   = 45.0     # °C

In [ ]:
# ── scenario definitions ──────────────────────────────────────────────────────
PEN_KEYS   = ["current_trend", "pniec", "entso_e"]
TEMP_KEYS  = ["baseline",      "ssp126", "ssp585"]
 
PEN_LABELS = {
    "current_trend": "Current Trend (30%)",
    "pniec":         "PNIEC 2024 (45%)",
    "entso_e":       "ENTSO-E DE (60%)",
}
TEMP_LABELS = {
    "baseline": "Baseline",
    "ssp126":   "SSP1-2.6",
    "ssp585":   "SSP5-8.5",
}
 
# 9-combination matrix
COMBINATIONS = [
    {"scenario_id": i*3+j,
     "pen_key":  pk,
     "temp_key": tk,
     "label":    f"{PEN_LABELS[pk]} | {TEMP_LABELS[tk]}"}
    for i, pk in enumerate(PEN_KEYS)
    for j, tk in enumerate(TEMP_KEYS)
]
 
# reference scenario: (pniec, baseline)
REF_PEN  = "pniec"
REF_TEMP = "baseline"
REF_ID   = next(c["scenario_id"] for c in COMBINATIONS
                if c["pen_key"]==REF_PEN and c["temp_key"]==REF_TEMP)
 
# visual settings
PEN_COLORS  = {"current_trend":"goldenrod", "pniec":"steelblue",
               "entso_e":"crimson"}
TEMP_STYLES = {"baseline":"-", "ssp126":"--", "ssp585":"-."}
 
print("── Step 0: Configuration ────────────────────────────────────────")
print(f"  G_STC={G_STC}  T_STC={T_STC}°C  γ={GAMMA}  PR={PR}  NOCT={NOCT}°C")
print(f"  Penetration scenarios : {PEN_KEYS}")
print(f"  Temperature scenarios : {TEMP_KEYS}")
print(f"  Total combinations   : {len(COMBINATIONS)}")
print(f"  Reference scenario   : ({REF_PEN}, {REF_TEMP})  id={REF_ID}")
print(f"\n  Combination matrix:")
for c in COMBINATIONS:
    ref = " ← REFERENCE" if c["scenario_id"] == REF_ID else ""
    print(f"    [{c['scenario_id']}] {c['label']}{ref}")